# Adaptive Neighborhood Exploration with GeniePath

Node Classification on PPI: Gated path-based neighborhood exploration using LSTM memory cells. This notebook ports the original PyTorch Geometric `GeniePathLazy` reference implementation to K3-Node: a `K3Breadth` module attends over each node's neighborhood with `GATConv`, and a `K3Depth` module (an `LSTMCell`) adaptively gates how far that information propagates across `layer_num` stacked layers. The single code cell below installs **K3-Node**, loads the PPI dataset, defines `K3GeniePathLazy` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops
import numpy as np

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import PPI
from k3_node.loader import DataLoader

title = "Adaptive Neighborhood Exploration with GeniePath on PPI"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset & Loaders
train_dataset = PPI(root="./data/PPI", split="train")
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

in_channels = train_dataset.num_features
out_channels = train_dataset.num_classes

# 2. GeniePath architecture: adaptive breadth (GAT) / depth (LSTM) exploration
dim = 256
lstm_hidden = 256
layer_num = 4

class K3Breadth(layers.Layer):
    """Breadth function: attends over a node's 1-hop neighborhood."""

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.gat = k3_layers.GATConv(in_dim, out_dim, heads=1)

    def call(self, x, edge_index):
        return ops.tanh(self.gat(x, edge_index))


class K3Depth(layers.Layer):
    """Depth function: an LSTM cell that gates how far to propagate."""

    def __init__(self, hidden):
        super().__init__()
        self.lstm_cell = layers.LSTMCell(hidden, use_bias=False)

    def call(self, x, h, c):
        out, (h, c) = self.lstm_cell(x, states=[h, c])
        return out, (h, c)


class K3GeniePathLazy(keras.Model):
    """Lazy GeniePath: every breadth function runs first over the same input,
    then each depth-wise LSTM unit consumes one breadth output in sequence."""

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin1 = layers.Dense(dim)
        self.breadths = [K3Breadth(dim, dim) for _ in range(layer_num)]
        self.depths = [K3Depth(lstm_hidden) for _ in range(layer_num)]
        self.lin2 = layers.Dense(out_dim)

    def call(self, inputs, edge_index=None):
        if isinstance(inputs, (tuple, list)):
            x, edge_index = inputs[0], inputs[1]
        else:
            x = inputs
        x = self.lin1(x)

        num_nodes = ops.shape(x)[0]
        h = ops.zeros((num_nodes, lstm_hidden))
        c = ops.zeros((num_nodes, lstm_hidden))

        h_tmps = [breadth(x, edge_index) for breadth in self.breadths]
        for breadth_out, depth in zip(h_tmps, self.depths):
            in_cat = ops.concatenate([breadth_out, x], axis=-1)
            x, (h, c) = depth(in_cat, h, c)
        return self.lin2(x)

k3_model = K3GeniePathLazy(in_channels, out_channels)

# 3. Model Compilation (Multi-label BCE)
k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.BinaryAccuracy(name="acc")],
)

# 4. Generator
def to_np(t, dtype=None):
    if hasattr(t, "cpu"):
        t = t.cpu()
    if hasattr(t, "detach"):
        t = t.detach()
    if hasattr(t, "numpy") and callable(t.numpy):
        t = t.numpy()
    return np.asarray(t, dtype=dtype)

def make_generator(loader):
    while True:
        for batch in loader:
            x = to_np(batch.x, dtype=np.float32)
            edge_index = to_np(batch.edge_index, dtype=np.int64)
            y = to_np(batch.y, dtype=np.float32)
            yield (x, edge_index), y

print(f"Training K3-Node GeniePath model on {backend} backend...")
history = k3_model.fit(
    make_generator(train_loader),
    steps_per_epoch=len(train_loader),
    epochs=10,
    verbose=1,
)

print("\n✓ K3-Node execution completed successfully!")